# Diffusion Models

This notebook accompanies the **ML Viz** lesson on diffusion models.
We'll implement a simple diffusion process and learn to reverse it.

**Companion lesson:** https://ml-viz.vercel.app/courses/generative-models/05-diffusion-models

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## The Forward Process (Adding Noise)

The forward process gradually adds Gaussian noise to data:

$$x_t = \sqrt{1 - \beta_t} \cdot x_{t-1} + \sqrt{\beta_t} \cdot \epsilon_t$$

After $T$ steps, the data is pure noise. Let's visualize this on 2D data.

In [ ]:
np.random.seed(42)

# Create 2D data: a circle
n = 200
theta = np.linspace(0, 2 * np.pi, n)
X0 = np.column_stack([2 * np.cos(theta), 2 * np.sin(theta)]) + 0.1 * np.random.randn(n, 2)

# Define noise schedule
T = 50
betas = np.linspace(0.0001, 0.02, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

def add_noise(x0, t):
    """Add noise to x0 at timestep t."""
    ab = alpha_bars[t]
    noise = np.random.randn(*x0.shape)
    return np.sqrt(ab) * x0 + np.sqrt(1 - ab) * noise, noise

# Show snapshots
timesteps = [0, 5, 15, 30, 49]
fig, axes = plt.subplots(1, len(timesteps), figsize=(4 * len(timesteps), 4))
fig.suptitle('Forward Process: Adding Noise', color='white', fontsize=13, y=1.02)

for ax, t in zip(axes, timesteps):
    if t == 0:
        xt = X0
    else:
        xt, _ = add_noise(X0, t)
    ax.scatter(xt[:, 0], xt[:, 1], c='#818cf8', s=10, alpha=0.6)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ab = alpha_bars[t]
    ax.set_title(f't={t}  ($\\bar{{\\alpha}}$={ab:.3f})', color='white', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Noise schedule

The $\beta_t$ schedule controls how quickly noise is added.
$\bar{\alpha}_t = \prod_{s=1}^t (1 - \beta_s)$ is the cumulative signal remaining.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(betas, color='#f43f5e', linewidth=2)
axes[0].set_title('$\\beta_t$ (noise rate per step)', color='white', fontsize=12)
axes[0].set_xlabel('Timestep $t$')

axes[1].plot(alpha_bars, color='#818cf8', linewidth=2)
axes[1].set_title('$\\bar{\\alpha}_t$ (signal remaining)', color='white', fontsize=12)
axes[1].set_xlabel('Timestep $t$')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f'At t=0:  signal = {alpha_bars[0]:.4f} (almost clean)')
print(f'At t={T-1}: signal = {alpha_bars[-1]:.6f} (almost pure noise)')

## The Reverse Process (Denoising)

The reverse process learns to remove noise step by step.
For our simple 2D data, we can learn the denoising function directly.

The key equation for each reverse step:

$$x_{t-1} = \frac{1}{\sqrt{1 - \beta_t}} \left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t)\right) + \sigma_t z$$

In [ ]:
class SimpleDenoiser:
    """Learns to predict noise added at each timestep."""
    
    def __init__(self):
        # Per-timestep linear denoisers (simplified)
        self.W = {}  # weight per timestep
        self.b = {}  # bias per timestep
    
    def predict_noise(self, x, t):
        """Predict the noise in x at timestep t."""
        if t in self.W:
            return x @ self.W[t] + self.b[t]
        return np.zeros_like(x)  # fallback
    
    def train(self, X0, T, betas, alphas, alpha_bars, epochs=200, lr=0.01):
        """Train denoiser on all timesteps."""
        losses = []
        for epoch in range(epochs):
            epoch_loss = 0
            for t in range(1, T):
                # Add noise
                ab = alpha_bars[t]
                noise = np.random.randn(*X0.shape)
                Xt = np.sqrt(ab) * X0 + np.sqrt(1 - ab) * noise
                
                # Predict noise
                W = np.zeros((2, 2))
                b = np.zeros(2)
                
                # Simple linear model: noise_pred = Xt @ W + b
                pred = Xt @ W + b
                
                # MSE loss
                err = pred - noise
                loss = np.mean(err ** 2)
                
                # Gradient descent
                dW = Xt.T @ err / len(X0)
                db = err.mean(axis=0)
                W -= lr * dW
                b -= lr * db
                
                self.W[t] = W.copy()
                self.b[t] = b.copy()
                epoch_loss += loss
            
            losses.append(epoch_loss / T)
            if (epoch + 1) % 50 == 0:
                print(f'Epoch {epoch+1:3d} | Avg Loss: {epoch_loss/T:.4f}')
        
        return losses

denoiser = SimpleDenoiser()
losses = denoiser.train(X0, T, betas, alphas, alpha_bars, epochs=200, lr=0.05)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color='#14b8a6', linewidth=1.5)
ax.set_title('Denoising Training Loss', color='white', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
plt.tight_layout()
plt.show()

## Sampling: Reverse the diffusion

Start from pure noise and iteratively denoise.

In [ ]:
def sample(denoiser, n_samples, T, betas, alpha_bars):
    """Generate samples by reversing the diffusion process."""
    x = np.random.randn(n_samples, 2)  # start from pure noise
    trajectory = [x.copy()]
    
    for t in range(T - 1, 0, -1):
        # Predict noise
        eps_pred = denoiser.predict_noise(x, t)
        
        # Reverse step
        beta_t = betas[t]
        ab = alpha_bars[t]
        sigma = np.sqrt(beta_t)
        
        x = (1 / np.sqrt(1 - beta_t)) * (x - (beta_t / np.sqrt(1 - ab)) * eps_pred)
        
        if t % 10 == 0:
            trajectory.append(x.copy())
    
    return x, trajectory

x_gen, trajectory = sample(denoiser, 200, T, betas, alpha_bars)

# Show the reverse process
fig, axes = plt.subplots(1, len(trajectory), figsize=(4 * len(trajectory), 4))
fig.suptitle('Reverse Process: Denoising to Generation', color='white', fontsize=13, y=1.02)

for ax, x_step in zip(axes, trajectory):
    ax.scatter(x_step[:, 0], x_step[:, 1], c='#14b8a6', s=10, alpha=0.6)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.axis('off')

axes[0].set_title('Pure Noise', color='white', fontsize=10)
axes[-1].set_title('Generated', color='white', fontsize=10)

plt.tight_layout()
plt.show()

## Real vs Generated

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X0[:, 0], X0[:, 1], c='#818cf8', s=15, alpha=0.6)
axes[0].set_title('Real Data', color='white', fontsize=12)
axes[0].set_xlim(-4, 4)
axes[0].set_ylim(-4, 4)
axes[0].set_aspect('equal')

axes[1].scatter(x_gen[:, 0], x_gen[:, 1], c='#14b8a6', s=15, alpha=0.6)
axes[1].set_title('Generated (Diffusion)', color='white', fontsize=12)
axes[1].set_xlim(-4, 4)
axes[1].set_ylim(-4, 4)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

## Key takeaways

1. **Forward process**: gradually add noise until data becomes pure Gaussian
2. **Reverse process**: learn to denoise step by step
3. **Training**: predict the noise $\epsilon_\theta(x_t, t)$ — simple MSE loss
4. **Sampling**: iterative — slower than GANs but higher quality and diversity
5. **Connection to score matching**: the denoiser learns $\nabla_x \log P(x)$

This is the foundation behind DALL·E, Stable Diffusion, and Midjourney.

## ✏️ Your turn

### Exercise 1 — Forward diffusion noise level

The closed-form forward process jumps directly to timestep $t$:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \varepsilon, \quad \varepsilon \sim \mathcal{N}(0,I)$$

where $\bar\alpha_t = \prod_{s=1}^{t}(1-\beta_s)$.  
As $t \to T$, $\bar\alpha_t \to 0$ and the signal is overwhelmed by noise.
Verify the boundary conditions using the notebook's beta schedule.

In [ ]:
import numpy as np

T_steps = 50
betas = np.linspace(0.0001, 0.02, T_steps)
alpha_bars = np.cumprod(1 - betas)   # reuse the notebook's schedule

def add_noise(x0, t, seed=0):
    """Add noise to x0 at timestep t (0-indexed) using the closed-form formula.
    x0: array, t: int, returns noisy array same shape as x0."""
    # TODO(you): sample eps ~ N(0,1) then apply the closed-form one-step jump
    ...

In [ ]:
# Boundary conditions on alpha_bars
assert alpha_bars[0] > 0.99, \
    "alpha_bar at t=0 must be ~1 (almost no noise yet)"
assert alpha_bars[-1] < 0.05, \
    "alpha_bar at t=T-1 must be ~0 (mostly noise)"

x0 = np.array([[2.0, 0.0]])
x_t0 = add_noise(x0, t=0, seed=0)
x_tT = add_noise(x0, t=49, seed=0)

assert np.linalg.norm(x_t0 - x0) < 0.5, \
    "at t=0 the noisy sample should be close to x0 (small beta)"
assert np.linalg.norm(x_tT - x0) > 1.0, \
    "at t=T-1 the noisy sample is far from x0 (signal destroyed)"
assert add_noise(x0, t=0, seed=0).shape == x0.shape, \
    "output shape must match x0"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def add_noise(x0, t, seed=0):
    rng = np.random.RandomState(seed)
    eps = rng.randn(*x0.shape)
    return np.sqrt(alpha_bars[t]) * x0 + np.sqrt(1 - alpha_bars[t]) * eps
```

</details>

### Exercise 2 — Reverse (denoising) step

Given the predicted noise $\hat\varepsilon = \varepsilon_\theta(x_t, t)$, the reverse step is:

$$x_{t-1} = \frac{1}{\sqrt{1-\beta_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\hat\varepsilon\right) + \sigma_t z,\quad z \sim \mathcal{N}(0,I)$$

where $\sigma_t = \sqrt{\beta_t}$. Implement this and verify the deterministic (z=0) case.

In [ ]:
def reverse_step(x_t, t, eps_hat, seed=0, stochastic=True):
    """One reverse diffusion step from x_t → x_{t-1}.
    t: current timestep (1-indexed here, so t=1 is the final step).
    eps_hat: predicted noise array same shape as x_t.
    If stochastic=False, sets z=0 (deterministic DDIM-like)."""
    # TODO(you): implement the reverse step formula
    # beta_t = betas[t-1], alpha_bar_t = alpha_bars[t-1]
    ...

In [ ]:
x_T = np.array([[0.5, -0.3]])
t_step = 25
# Use a zero noise predictor (eps_hat = 0) for easy hand-checking
x_prev_det = reverse_step(x_T, t_step, eps_hat=np.zeros_like(x_T), stochastic=False)

assert x_prev_det.shape == x_T.shape, "output shape must equal input shape"

# With eps_hat=0, the formula reduces to x_T / sqrt(1 - beta_t)
beta_t = betas[t_step - 1]
expected = x_T / np.sqrt(1 - beta_t)
assert np.allclose(x_prev_det, expected, atol=1e-9), \
    "with zero predicted noise, denoising step = x_t / sqrt(1-beta_t)"

# Stochastic step has different output from deterministic (due to added noise)
np.random.seed(1)
x_prev_sto = reverse_step(x_T, t_step, eps_hat=np.zeros_like(x_T), stochastic=True)
assert not np.allclose(x_prev_det, x_prev_sto, atol=1e-6), \
    "stochastic and deterministic steps should differ when sigma_t > 0"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def reverse_step(x_t, t, eps_hat, seed=0, stochastic=True):
    beta_t      = betas[t - 1]
    alpha_bar_t = alpha_bars[t - 1]
    coeff       = beta_t / np.sqrt(1 - alpha_bar_t)
    x_prev      = (x_t - coeff * eps_hat) / np.sqrt(1 - beta_t)
    if stochastic:
        rng = np.random.RandomState(seed)
        x_prev = x_prev + np.sqrt(beta_t) * rng.randn(*x_t.shape)
    return x_prev
```

</details>